# Query Decomposition

**Query Decomposition** is a retrieval technique that improves Retrieval-Augmented Generation (RAG) by breaking a complex user question into multiple smaller and easier-to-answer sub-questions. Each sub-question is answered individually using retrieval, and the results are combined to generate a comprehensive final answer.

Unlike traditional RAG, which retrieves documents for a single query, Query Decomposition enables the system to reason through complex or multi-hop questions by solving smaller problems first.

---

## Why Query Decomposition?

Some questions contain multiple concepts or require reasoning across several pieces of information. A single retrieval step may not capture all the necessary context.

Query Decomposition addresses this by:

- Breaking complex questions into simpler sub-questions.
- Retrieving relevant context for each sub-question.
- Answering each sub-question individually.
- Combining all intermediate answers into a final response.

---

## Workflow

```text
            User Question
                  │
                  ▼
      Generate Sub-Questions
                  │
      ┌───────────┼───────────┐
      ▼           ▼           ▼
 Sub-Question 1  Sub-Question 2  Sub-Question 3
      │           │           │
      ▼           ▼           ▼
 Retrieve Docs Retrieve Docs Retrieve Docs
      │           │           │
      ▼           ▼           ▼
    Answer 1    Answer 2    Answer 3
          \        |        /
           \       |       /
            ▼      ▼      ▼
          Combine Answers
                  │
                  ▼
          Final Response
```

---

## Types of Query Decomposition

### 1. Sequential Query Decomposition

Sub-questions are answered one after another. Each newly generated answer is used as additional context for the next sub-question, allowing the model to build knowledge progressively.

### 2. Parallel Query Decomposition

All sub-questions are answered independently. Their individual answers are then combined to produce the final response. This approach is faster and works well when the sub-questions are independent.

---

## Advantages

- Improves retrieval for complex and multi-hop questions.
- Breaks difficult problems into simpler tasks.
- Provides more comprehensive and accurate answers.
- Reduces the chance of missing relevant information.
- Enables better reasoning over multiple pieces of retrieved context.

---

## One-Line Definition

> **Query Decomposition is a RAG technique that splits a complex question into multiple simpler sub-questions, retrieves information for each one, and combines their answers to generate a more accurate final response.**


## 1. Query Decomposition

**Query Decomposition** is a retrieval technique that breaks a complex user question into multiple smaller and easier-to-answer sub-questions. Each sub-question is processed individually, and their answers are combined to produce a more complete and accurate final response.

This approach is especially useful for complex, multi-step, or multi-hop questions that cannot be answered effectively with a single retrieval.


In [2]:
# Basic setup run gareko
%run ./pipeline/01_basic_rag.ipynb

# Indexing run gareko (documents, embeddings, vectorstore, retriever)
%run ./pipeline/02_indexing_rag.ipynb

1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558


In [3]:
from langchain_core.prompts import ChatPromptTemplate

# Query Decomposition ko lagi prompt template create gareko
template = """
You are a helpful assistant that generates multiple sub-questions related to an input question.

The goal is to break down the input into a set of sub-problems or sub-questions that can be answered independently.

Generate 3 sub-questions related to:

{question}

Output (3 queries):
"""

# Query Decomposition ko prompt template create gareko
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# Local Ollama LLM use gareko (OpenAI ko satta)
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Prompt → LLM → Output Parser → Sub-questions generate garne chain
generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    # Generate bhayeka sub-questions lai newline ko basis ma list ma split gareko
    | (lambda x: x.split("\n"))
)


# User ko complex question
question = "What are the main components of an LLM-powered autonomous agent system?"


# Query Decomposition chain execute gareko
questions = generate_queries_decomposition.invoke({"question": question})


# Generate bhayeka sub-questions print garne
print(questions)

['Here are three sub-questions related to "What are the main components of an LLM-powered autonomous agent system?":', '', '1. What is the role of Large Language Models (LLMs) in an autonomous agent system, and how do they contribute to decision-making?', '2. What are some key architectural components that enable an LLM-powered autonomous agent system to perceive its environment, make decisions, and take actions?', '3. How do LLMs interact with other AI/ML models or systems within the autonomous agent framework, such as computer vision, robotics, or sensor processing modules?']


## 2. Sequential Query Decomposition

In **Sequential Query Decomposition**, sub-questions are answered one after another. The answer to each previous sub-question is used as additional context for answering the next one. This allows the model to build knowledge progressively and reason through complex problems step by step.

This approach is beneficial when later sub-questions depend on information obtained from earlier ones.


In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Final Query Decomposition ko lagi prompt template create gareko
template = """
Here is the question you need to answer:

--------------------
{question}
--------------------

Here are the available background question and answer pairs:

--------------------
{q_a_pairs}
--------------------

Here is additional context relevant to the question:

--------------------
{context}
--------------------

Use the retrieved context and the background question-answer pairs to answer the original question.

Answer:
"""

# Final prompt template create gareko
decomposition_prompt = ChatPromptTemplate.from_template(template)

In [6]:
from operator import itemgetter

from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser


# Question ra answer pair lai format garne function
def format_qa_pair(question, answer):
    formatted_string = ""
    formatted_string += f"Question: {question}\n"
    formatted_string += f"Answer: {answer}\n\n"
    return formatted_string.strip()


# Local Ollama LLM use gareko (OpenAI ko satta)
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Previous question-answer pairs store garna variable
q_a_pairs = ""


# Generate bhayeka pratyek sub-question ko lagi iterate garne
for q in questions:

    # Pratyek sub-question ko lagi RAG chain create gareko
    rag_chain = (
        {
            # Retriever bata relevant context retrieve garne
            "context": itemgetter("question") | retriever,
            # Current sub-question prompt ma pathaune
            "question": itemgetter("question"),
            # Previous Q&A pairs prompt ma pathaune
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # Prompt template ma values inject garne
        | decomposition_prompt
        # Local Ollama LLM bata answer generate garne
        | llm
        # Output lai plain text ma convert garne
        | StrOutputParser()
    )

    # Current sub-question ko answer generate gareko
    answer = rag_chain.invoke(
        {
            "question": q,
            "q_a_pairs": q_a_pairs,
        }
    )

    # Current Question-Answer pair format gareko
    q_a_pair = format_qa_pair(q, answer)

    # Future sub-questions ko lagi Q&A history update gareko
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair


# Final Q&A pairs print garne
print(q_a_pairs)


---
Question: Here are three sub-questions related to "What are the main components of an LLM-powered autonomous agent system?":
Answer: Based on the provided context and background question-answer pairs, the main components of an LLM-powered autonomous agent system are:

1. **Planning**: This component involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. Additionally, the agent can reflect on past actions, learn from mistakes, and refine them for future steps to improve the quality of final results.
2. **Memory**: The memory component includes short-term memory, which refers to all in-context learning.

These components complement the LLM (large language model) as the core controller of the autonomous agent system, enabling it to function as a powerful general problem solver.
---
Question: 
Answer: Based on the provided context and background question-answer pairs, the main components of an LLM-powered autonomous agent sy

## 3. Parallel Query Decomposition

In **Parallel Query Decomposition**, all sub-questions are answered independently. Each sub-question retrieves its own relevant documents and generates an answer without relying on other sub-question answers. The individual answers are then combined to produce the final response.

This approach is faster and works well when the sub-questions are independent of one another.


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# Local Ollama LLM use gareko
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Local RAG prompt create gareko (LangChain Hub ko satta)
prompt_rag = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.

Context:
{context}

Question:
{question}

Answer:
""")


# Pratyek sub-question ko lagi Retrieval + RAG garne function
def retrieve_and_rag(question, prompt_rag, sub_question_generator_chain):

    # Original question lai sub-questions ma divide gareko
    sub_questions = sub_question_generator_chain.invoke({"question": question})

    # Pratyek sub-question ko answer store garna list
    rag_results = []

    # Pratyek sub-question ko lagi iterate garne
    for sub_question in sub_questions:

        # Sub-question ko lagi relevant documents retrieve gareko
        retrieved_docs = retriever.invoke(sub_question)

        # Retrieved context use garera answer generate gareko
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {
                "context": retrieved_docs,
                "question": sub_question,
            }
        )

        # Generated answer list ma add gareko
        rag_results.append(answer)

    # Sabai answers ra sub-questions return gareko
    return rag_results, sub_questions


# Query Decomposition + Retrieval + RAG execute gareko
answers, questions = retrieve_and_rag(
    question,
    prompt_rag,
    generate_queries_decomposition,
)


# Generate bhayeka answers print garne
print(answers)

['According to the provided context, the main components of an LLM-powered autonomous agent system include:\n\n1. Planning: This component involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\n2. Reflection and refinement: This component allows the agent to do self-criticism and self-reflection over past actions, learn from mistakes, and refine them for future steps, thereby improving the quality of final results.\n\nNote that there is no mention of a third main component in the provided context.', "Based only on the provided context, I can't answer the question because there is no specific question being asked. The text appears to be a set of instructions and guidelines for an AI assistant, but it doesn't contain a specific question that needs to be answered. If you'd like me to help with something else, feel free to ask!", 'According to the provided context, LLMs function as the "agent\'s brain" in a LLM-powered autonomou

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Sabai Question-Answer pairs lai string format ma convert garne function
def format_qa_pairs(questions, answers):
    formatted_string = ""

    # Pratyek Question ra Answer pair iterate garne
    for i, (question, answer) in enumerate(
        zip(questions, answers),
        start=1,
    ):
        formatted_string += f"Question {i}: {question}\n" f"Answer {i}: {answer}\n\n"

    return formatted_string.strip()


# Sabai Q&A pairs lai final context banaune
context = format_qa_pairs(questions, answers)


# Final answer generate garna prompt template create gareko
template = """
Here is a set of Question and Answer pairs:

{context}

Use these Question-Answer pairs to generate the final answer to the following question.

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


# Final synthesis chain create gareko
final_rag_chain = (
    prompt
    # Local Ollama LLM use garera final answer generate garne
    | llm
    # Output lai plain text ma convert garne
    | StrOutputParser()
)


# Original question ko final synthesized answer generate gareko
response = final_rag_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)


# Final answer print garne
print(response)

Based on the provided context, the main components of an LLM-powered autonomous agent system include:

1. Planning: This component involves breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
2. Reflection and refinement: This component allows the agent to do self-criticism and self-reflection over past actions, learn from mistakes, and refine them for future steps.

Note that there is no mention of a third main component in the provided context.
